# Thai Word Segmentation - CRF + AttaCut Ensemble

Notebook นี้:
1. เทรน CRF บน LST20 corpus (character-level BMES)
2. ใช้ AttaCut (Bi-LSTM) เป็น second opinion
3. Ensemble ด้วย **majority vote** ระดับ character tag → B_WORD/I_WORD/E_WORD

## 1) Install dependencies

In [ ]:
# Install attacut without downgrading numpy: override nptyping compat
# --no-deps prevents numpy downgrade; we install attacut's other deps manually
!pip install -q sklearn-crfsuite
!pip install -q attacut --no-deps
# Install attacut's actual deps (except numpy) so it works
!pip install -q docopt fire ssg pythainlp
# nptyping compatibility shim - install but numpy stays at 2.x
!pip install -q 'nptyping>=2.5.0'

## 2. Configure File Paths

In [ ]:
DATA_DIR = "/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation"
DATASET_DIR = "/kaggle/input/datasets/guntinunsawatvong/lst20-corpus-guntinun"

In [ ]:
from pathlib import Path
import os

def detect_path(base_dir, candidates, must_exist=True):
    for name in candidates:
        p = base_dir / name
        if p.exists():
            return p
    if must_exist:
        raise FileNotFoundError(f"Cannot find any of {candidates} in {base_dir}")
    return None

data_dir = Path(DATA_DIR)
test_path = detect_path(data_dir, ["ws_test.txt", "test.txt"])
sample_path = detect_path(data_dir, ["ws_sample_submission.csv", "sample_submission.csv"])
print(f"Test path:   {test_path}")
print(f"Sample path: {sample_path}")

# Find LST20 corpus directory
lst20_root = Path(DATASET_DIR)
lst20_candidates = list(lst20_root.rglob('LST20_Corpus'))
lst20_dir = lst20_candidates[0] if lst20_candidates else lst20_root
print(f"LST20 dir:   {lst20_dir}")

## 3. Parse LST20 Corpus → Training Sentences
อ่านไฟล์ CoNLL จาก LST20 corpus แล้วแปลงเป็น list ของประโยค (แต่ละประโยค = list ของคำ)


In [ ]:
def parse_lst20_conll(filepath):
    sentences = []
    current = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.rstrip('\n')
                if not line or line.startswith('#'):
                    if current:
                        sentences.append(current)
                        current = []
                else:
                    parts = line.split('\t')
                    word = parts[0]
                    if word and word != '_':
                        current.append(word)
        if current:
            sentences.append(current)
    except Exception as e:
        print(f"Skip {filepath}: {e}")
    return sentences

# Load TRAIN split
print("Loading train split...")
train_sentences = []
train_dir = lst20_dir / 'train'
if train_dir.exists():
    for fp in sorted(train_dir.rglob('*.txt')):
        train_sentences.extend(parse_lst20_conll(fp))
print(f"Train sentences: {len(train_sentences)}")

# Load EVAL split too (we want as much in-domain data as possible)
print("Loading eval split...")
eval_sentences = []
eval_dir = lst20_dir / 'eval'
if eval_dir.exists():
    for fp in sorted(eval_dir.rglob('*.txt')):
        eval_sentences.extend(parse_lst20_conll(fp))
print(f"Eval sentences:  {len(eval_sentences)}")

# Merge both for maximum training signal
all_sentences = train_sentences + eval_sentences
print(f"Total training sentences: {len(all_sentences)}")
print(f"Sample[0]: {all_sentences[0][:10] if all_sentences else 'EMPTY'}")

## 4. Train CRF Model on LST20
เทรน CRF บน features ระดับอักขระจาก LST20 (BMES tagging scheme)


In [ ]:
import sklearn_crfsuite
from tqdm.auto import tqdm

def char_type(c):
    code = ord(c)
    if 0x0E01 <= code <= 0x0E2E: return 'TH_C'
    if 0x0E30 <= code <= 0x0E3A: return 'TH_V'
    if 0x0E40 <= code <= 0x0E44: return 'TH_LV'
    if 0x0E48 <= code <= 0x0E4B: return 'TH_T'
    if '\u0E00' <= c <= '\u0E7F': return 'TH_O'
    if c.isdigit(): return 'D'
    if c.isalpha(): return 'A'
    return 'P'

def char2features(chars, i):
    c = chars[i]
    n = len(chars)
    f = {'bias': 1.0, 'c[0]': c, 'type': char_type(c)}

    if i >= 1:
        f['c[-1]'] = chars[i-1]; f['c[-1:0]'] = chars[i-1]+c; f['t[-1]'] = char_type(chars[i-1])
    else:
        f['BOS'] = True
    if i >= 2:
        f['c[-2]'] = chars[i-2]; f['c[-2:-1]'] = chars[i-2]+chars[i-1]; f['c[-2:0]'] = chars[i-2]+chars[i-1]+c
    if i >= 3:
        f['c[-3]'] = chars[i-3]; f['c[-3:-1]'] = chars[i-3]+chars[i-2]+chars[i-1]
    if i >= 4:
        f['c[-4]'] = chars[i-4]

    if i < n-1:
        f['c[+1]'] = chars[i+1]; f['c[0:+1]'] = c+chars[i+1]; f['t[+1]'] = char_type(chars[i+1])
    else:
        f['EOS'] = True
    if i < n-2:
        f['c[+2]'] = chars[i+2]; f['c[+1:+2]'] = chars[i+1]+chars[i+2]; f['c[0:+2]'] = c+chars[i+1]+chars[i+2]
    if i < n-3:
        f['c[+3]'] = chars[i+3]; f['c[+1:+3]'] = chars[i+1]+chars[i+2]+chars[i+3]
    if i < n-4:
        f['c[+4]'] = chars[i+4]

    # Thai-specific: leading vowel before current = likely word start
    if i >= 1 and 0x0E40 <= ord(chars[i-1]) <= 0x0E44:
        f['prev_is_lead_vowel'] = True
    # Current is tone mark = definitely not word start
    if 0x0E48 <= ord(c) <= 0x0E4B:
        f['is_tone'] = True

    return f

def sent2features(words):
    chars = list(''.join(words))
    return [char2features(chars, i) for i in range(len(chars))]

def sent2labels(words):
    labels = []
    for w in words:
        n = len(w)
        if n == 1:   labels.append('S')
        elif n == 2: labels.extend(['B','E'])
        else:
            labels.append('B'); labels.extend(['M']*(n-2)); labels.append('E')
    return labels

sents = all_sentences
print(f"Building features for {len(sents)} sentences...")
X_train, y_train = [], []
for s in tqdm(sents, desc="Feature extraction", unit="sent"):
    X_train.append(sent2features(s))
    y_train.append(sent2labels(s))
print(f"Done. Total sequences: {len(X_train)}")

# Build LST20 vocabulary for post-processing
vocab = set()
for sent in all_sentences:
    for word in sent:
        vocab.add(word)
print(f"LST20 vocabulary size: {len(vocab)}")

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.02,
    c2=0.02,
    max_iterations=300,
    all_possible_transitions=True,
    verbose=True
)
print("Training CRF (300 iterations, ±4 window)...")
crf.fit(X_train, y_train)
print("CRF training complete!")


## 5. CRF Inference on Test Text


In [ ]:
test_text_raw = test_path.read_text(encoding='utf-8')
test_text = test_text_raw.replace(' ', '')
print(f"Test text length: {len(test_text)}")

# ── CRF inference in chunks ──────────────────────────────────────────
CHUNK = 2000
chars = list(test_text)
crf_label_seq = []

for start in tqdm(range(0, len(chars), CHUNK), desc="CRF inference"):
    chunk_chars = chars[start:start+CHUNK]
    X_chunk = [char2features(chunk_chars, i) for i in range(len(chunk_chars))]
    pred = crf.predict_single(X_chunk)
    crf_label_seq.extend(pred)

print(f"CRF labels: {len(crf_label_seq)}")

# ── BMES → word list ────────────────────────────────────────────────
def bmes_to_words(chars, labels):
    """Convert BMES label sequence to list of words."""
    raw_words = []
    current = ''
    for ch, label in zip(chars, labels):
        current += ch
        if label in ('E', 'S'):
            raw_words.append(current)
            current = ''
    if current:
        raw_words.append(current)
    return raw_words

crf_raw_words = bmes_to_words(chars, crf_label_seq)

# ── Vocabulary-based post-processing ────────────────────────────────
def vocab_postprocess(words, vocab, max_merge=4):
    result = []
    i = 0
    while i < len(words):
        merged = False
        for j in range(min(max_merge, len(words)-i), 1, -1):
            candidate = ''.join(words[i:i+j])
            pieces = words[i:i+j]
            has_short_piece = any(len(p) <= 2 for p in pieces[:-1])
            if candidate in vocab and has_short_piece:
                result.append(candidate)
                i += j
                merged = True
                break
        if not merged:
            result.append(words[i])
            i += 1
    return result

crf_words = vocab_postprocess(crf_raw_words, vocab, max_merge=4)
print(f"CRF words before pp: {len(crf_raw_words)}, after pp: {len(crf_words)}")

# ── Convert CRF words → character-level BIE labels ──────────────────
def words_to_bie_labels(words):
    """Convert word list to character-level B/I/E label list."""
    labels = []
    for word in words:
        n = len(word)
        if n == 1:
            labels.append('B')  # single-char word: treat as B (=B_WORD)
        elif n == 2:
            labels.extend(['B', 'E'])
        else:
            labels.append('B')
            labels.extend(['I'] * (n - 2))
            labels.append('E')
    return labels

crf_bie = words_to_bie_labels(crf_words)
print(f"CRF BIE labels: {len(crf_bie)} (expected {len(test_text)})")

## 6. AttaCut Inference
ใช้ AttaCut (Bi-LSTM character-level model) เป็น second model สำหรับ ensemble

**Note on numpy compatibility:** AttaCut 1.0.6 ต้องการ nptyping ซึ่งมี annotation เก่า  
เราจะ suppress warnings และ patch import เพื่อให้ทำงานกับ numpy 2.x ได้

In [ ]:
import warnings
import sys

# Patch nptyping to work with numpy 2.x
# nptyping uses np.bool, np.int, np.float etc. which are removed in numpy 2.x
import numpy as np
print(f"NumPy version: {np.__version__}")

# Add compatibility shims for old numpy attributes
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'complex'):
    np.complex = complex
if not hasattr(np, 'object'):
    np.object = object
if not hasattr(np, 'str'):
    np.str = str

# Now import attacut with warnings suppressed
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        from attacut import Tokenizer as AttaCutTokenizer
        attacut_tok = AttaCutTokenizer(model="attacut-sc")
        print("AttaCut loaded successfully!")
        ATTACUT_OK = True
    except Exception as e:
        print(f"AttaCut load failed: {e}")
        ATTACUT_OK = False

In [ ]:
if ATTACUT_OK:
    # AttaCut tokenizes text, returns word list
    # Process in chunks to avoid memory issues
    CHUNK_SIZE = 5000  # chars per chunk
    attacut_words_all = []
    
    for start in tqdm(range(0, len(test_text), CHUNK_SIZE), desc="AttaCut inference"):
        chunk = test_text[start:start+CHUNK_SIZE]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            tokens = attacut_tok.tokenize(chunk)
        attacut_words_all.extend(tokens)
    
    total_attacut_chars = sum(len(w) for w in attacut_words_all)
    print(f"AttaCut word count: {len(attacut_words_all)}")
    print(f"AttaCut total chars: {total_attacut_chars} (expected {len(test_text)})")
    
    if total_attacut_chars != len(test_text):
        print("WARNING: char count mismatch! AttaCut may have altered text.")
        # Attempt to realign by reconstructing from chars
        attacut_joined = ''.join(attacut_words_all)
        print(f"  Joined length: {len(attacut_joined)}")
    
    attacut_bie = words_to_bie_labels(attacut_words_all)
    print(f"AttaCut BIE labels: {len(attacut_bie)}")
else:
    print("Skipping AttaCut - using CRF only")
    attacut_bie = None

## 7. Ensemble: CRF + AttaCut → Majority Vote

Strategy:
- แต่ละ character มี 2 votes: CRF และ AttaCut
- ถ้าตกลงกัน → ใช้ค่านั้น
- ถ้าไม่ตรงกัน → **เชื่อ CRF** (trained on in-domain LST20 data)
  แต่ถ้า AttaCut บอก `B` (word boundary) ให้ weight มากกว่า
  เพราะ AttaCut มีแนวโน้ม precision สูง

**Ensemble rule:**
- ถ้าทั้งคู่ตกลงกัน → ใช้นั้น
- ถ้าไม่ตรงกัน → ใช้ CRF (higher weight from in-domain training)

**Alternatively:** แปลง BIE labels ทั้งสองชุดกลับเป็น word boundary bits แล้ว OR/AND
- `BOUNDARY = B or S (single-char)` → bit = 1 (word starts here)
- Ensemble boundary: take **union** (if either says boundary → boundary)
  → This reduces under-segmentation
- Or **intersection** (both must agree on boundary)
  → This reduces over-segmentation

In [ ]:
def bie_to_boundary_bits(bie_labels):
    """Convert BIE label list to binary boundary bits.
    bit[i] = 1 means char[i] is the START of a new word.
    """
    return [1 if lbl == 'B' else 0 for lbl in bie_labels]

def boundary_bits_to_words(chars, bits):
    """Reconstruct word list from chars + boundary bits."""
    words = []
    current = ''
    for ch, bit in zip(chars, bits):
        if bit == 1 and current:
            words.append(current)
            current = ''
        current += ch
    if current:
        words.append(current)
    return words

def boundary_bits_to_bie(bits, n):
    """Convert boundary bits back to BIE labels (filling I for interior)."""
    bie = []
    for i in range(n):
        if bits[i] == 1:  # word start
            bie.append('B')
        else:
            # Check: is next char a word start or end of sequence?
            if i == n-1 or bits[i+1] == 1:
                bie.append('E')
            else:
                bie.append('I')
    return bie

crf_bits = bie_to_boundary_bits(crf_bie)

if ATTACUT_OK and attacut_bie is not None and len(attacut_bie) == len(crf_bie):
    attacut_bits = bie_to_boundary_bits(attacut_bie)
    
    # --- Strategy: weighted vote ---
    # CRF weight = 2 (stronger, in-domain trained)
    # AttaCut weight = 1
    # Boundary (B) if weighted_sum >= threshold
    # This is equivalent to: boundary if CRF says B, OR (both say B)
    # We try threshold = 1.5 meaning: CRF alone (weight 2) is enough
    # but AttaCut alone (weight 1) is NOT enough
    
    # Try different strategies and pick the most sensible one:
    
    # Strategy A: CRF primary, AttaCut adds boundaries (UNION)
    # Adds word boundaries that AttaCut sees but CRF misses
    # Could over-segment but may catch real boundaries CRF missed
    bits_union = [1 if (a or b) else 0 for a, b in zip(crf_bits, attacut_bits)]
    
    # Strategy B: Only agree on boundaries (INTERSECTION)
    # Most conservative - only boundaries both agree on
    bits_intersect = [1 if (a and b) else 0 for a, b in zip(crf_bits, attacut_bits)]
    
    # Strategy C: CRF wins, but AttaCut can veto non-boundaries
    # If CRF says B but AttaCut says not-B → still B (CRF wins)
    # If CRF says not-B but AttaCut says B → add boundary (union for additions only)
    # Actually this IS the union
    
    # Strategy D: Weighted - CRF=2, AttaCut=1, threshold=1.5
    # boundary if crf=1 (weight 2 → sum=2 ≥ 1.5) or both=1 (sum=3 ≥ 1.5)
    # = CRF only required (AttaCut ignored for adding boundaries)
    bits_weighted = [1 if (2*a + 1*b) >= 2 else 0 
                     for a, b in zip(crf_bits, attacut_bits)]
    # This = keep all CRF boundaries, ignore AttaCut-only boundaries
    # But AttaCut can REMOVE boundaries if CRF=0, AttaCut=0 (which is same as CRF)
    # Better: CRF=2, threshold=1 → always follow CRF, OR AttaCut alone
    
    # Best strategy for higher F1:
    # Add AttaCut boundaries where CRF is uncertain
    # Since we don't have CRF confidence scores, use UNION (adds) and INTERSECTION (removes)
    # Report counts:
    print(f"CRF boundaries:       {sum(crf_bits)}")
    print(f"AttaCut boundaries:   {sum(attacut_bits)}")
    print(f"UNION boundaries:     {sum(bits_union)}")
    print(f"INTERSECT boundaries: {sum(bits_intersect)}")
    print(f"Weighted (CRF-pri):   {sum(bits_weighted)}")
    
    # Use CRF-primary weighted approach (most conservative ensemble)
    # but allow AttaCut to add boundaries (union) for recall boost
    # Final choice: UNION strategy to boost recall
    final_bits = bits_union
    print("\nUsing: UNION ensemble (CRF + AttaCut boundaries)")
    
    # Ensure bit[0] = 1 (first char is always a word start)
    if final_bits[0] == 0:
        final_bits[0] = 1
    
    final_words = boundary_bits_to_words(chars, final_bits)
    final_bie = boundary_bits_to_bie(final_bits, len(chars))
    
    print(f"Final word count: {len(final_words)}")
    print(f"Final BIE labels: {len(final_bie)}")
    print(f"Sample words: {final_words[:20]}")

else:
    print("Using CRF-only (AttaCut unavailable or length mismatch)")
    if ATTACUT_OK and attacut_bie is not None:
        print(f"  Length mismatch: CRF={len(crf_bie)}, AttaCut={len(attacut_bie)}")
    final_bie = crf_bie
    final_words = crf_words
    print(f"CRF word count: {len(final_words)}")

## 8. Convert BIE → B_WORD / I_WORD / E_WORD Tags


In [ ]:
# Convert internal BIE labels → submission format
def bie_to_submission_labels(bie_labels):
    """Convert internal BIE → B_WORD/I_WORD/E_WORD for submission."""
    result = []
    for lbl in bie_labels:
        if lbl == 'B':
            result.append('B_WORD')
        elif lbl == 'I':
            result.append('I_WORD')
        elif lbl == 'E':
            result.append('E_WORD')
        else:
            result.append('B_WORD')  # fallback
    return result

pred_labels = bie_to_submission_labels(final_bie)

print(f"Total chars:  {len(test_text)}")
print(f"Total labels: {len(pred_labels)}")
print(f"Expected:     35182")
assert len(pred_labels) == len(test_text), \
    f"Label count mismatch: {len(pred_labels)} != {len(test_text)}"
print("✓ Label count matches!")

## 9. Export Submission CSV (using csv module - no pandas)


In [ ]:
import csv

out_path = Path("submission.csv")
print(f"Writing -> {out_path.resolve()}")

with open(sample_path, 'r', encoding='utf-8') as f_in, \
     open(out_path, 'w', encoding='utf-8', newline='') as f_out:
    reader = csv.reader(f_in)
    writer = csv.writer(f_out)
    headers = next(reader)
    writer.writerow(headers)
    for row_idx, row in enumerate(reader):
        row[1] = pred_labels[row_idx] if row_idx < len(pred_labels) else "B_WORD"
        writer.writerow(row)

# Verify output
with open(out_path, 'r') as f:
    lines = f.readlines()
print(f"Submission rows: {len(lines)-1} (expected 35182)")
print("First 4 rows:")
for line in lines[:5]:
    print(' ', line.strip())
print("\nSubmission created!")